# HELD: BCC force constants through 5NN, saved by MD step

This notebook scans the complete non-magnetic iron dataset under `IronCoreMD/dataset/bcc/non-mag`, fits every compatible Fe BCC trajectory with the repository's HELD implementation using five BCC neighbor shells, and saves one compressed NPZ per trajectory plus a master index. Magnetic trajectories and B2 materials are intentionally excluded.

For every MD step, the output stores the 14 traditional monoatomic-BCC Born–von Kármán elements through 5NN: `alpha_0`, `alpha_1`, `beta_1`, `alpha_2`, `beta_2`, `alpha_3`, `beta_3`, `gamma_3`, `alpha_4`, `beta_4`, `gamma_4`, `delta_4`, `alpha_5`, and `beta_5`. It also retains all HELD basis coefficients and complete onsite/offsite tensors. `alpha_0` is derived from the other 13 independent elements by the acoustic sum rule.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / 'held_bcc_5nn.py').is_file():
    raise FileNotFoundError('Run this notebook from IronCoreMD/ForceConstants')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from held_bcc_5nn import (
    BVK_LABELS, DATASET_ROOT, RESULTS_ROOT, FitConfig,
    discover_bcc_npz, fit_trajectory, run_dataset, summarize_index,
)

print('Dataset:', DATASET_ROOT)
print('Results:', RESULTS_ROOT)
print('14 BvK labels:', BVK_LABELS.tolist())

## 1. Discover and audit non-magnetic Fe BCC NPZ files

Only NPZ files inside `dataset/bcc/non-mag` with the complete HELD trajectory schema and elemental Fe labels are selected.

In [ ]:
trajectories, skipped = discover_bcc_npz()
inventory = pd.DataFrame({
    'relative_path': [str(path.relative_to(DATASET_ROOT)) for path in trajectories],
    'size_MB': [path.stat().st_size / 1024**2 for path in trajectories],
})
print(f'Compatible trajectories: {len(trajectories)}')
print(f'Skipped NPZ files: {len(skipped)}')
display(inventory)
display(pd.DataFrame(skipped))

## 2. Smoke test on one real trajectory

This fits two finite frames and validates the result schema before starting the full dataset. It writes into `results/smoke/` and does not overwrite production outputs.

In [ ]:
smoke_source = next(
    (path for path in trajectories if 'non-mag' in path.parts),
    trajectories[0],
)
smoke = fit_trajectory(
    smoke_source,
    config=FitConfig(skip=0, every=1, max_frames=2, overwrite=True),
    results_root=RESULTS_ROOT / 'smoke',
    verbose=True,
)
smoke

In [ ]:
with np.load(smoke['output_path'], allow_pickle=False) as result:
    expected = ['alpha_0', 'alpha_1', 'beta_1', 'alpha_2', 'beta_2', 'alpha_3', 'beta_3', 'gamma_3', 'alpha_4', 'beta_4', 'gamma_4', 'delta_4', 'alpha_5', 'beta_5']
    assert result['fc_labels'].tolist() == expected
    assert result['fc_per_md_step'].shape == (2, 14)
    assert result['step_ids'].shape == (2,)
    assert result['held_coefficients_per_frame'].shape[0] == 2
    assert result['offsite_fc_per_frame'].shape[0] == 2
    assert result['onsite_fc_per_frame'].shape[0] == 2
    display(pd.DataFrame(
        result['fc_per_md_step'],
        index=result['step_ids'],
        columns=result['fc_labels'],
    ).rename_axis('md_step'))
print('PASS: per-MD-step arrays and alpha labels validated')

## 3. Full non-magnetic Fe BCC dataset fit

Set `RUN_FULL_DATASET = True` when ready. The default configuration uses every finite MD frame and all five shells. This is computationally expensive because the repository contains many 128-atom, 400-frame trajectories. Completed case files are reused on restart.

To discard equilibration frames, change `skip`; to thin correlated frames, change `every`. Those selections are stored in every result file.

In [ ]:
RUN_FULL_DATASET = False  # Change to True to launch all compatible BCC trajectories.
CONFIG = FitConfig(
    skip=0,
    every=1,
    max_frames=0,  # 0 means all selected frames
    aggregate='mean',
    num_shells=5,
    overwrite=False,  # resume safely from existing per-case results
)

if RUN_FULL_DATASET:
    records, skipped, master_index = run_dataset(config=CONFIG, verbose=True)
    print('Master index:', master_index)
    summarize_index(master_index)
else:
    print('Dry run only. Set RUN_FULL_DATASET=True to fit the full BCC dataset.')

## 4. Inspect the master index and per-step results

The master index stores one row per attempted trajectory. Each `output_path` points to a detailed NPZ containing the step-resolved arrays.

In [ ]:
master_index = RESULTS_ROOT / 'bcc_held_5nn_index.npz'
if master_index.exists():
    with np.load(master_index, allow_pickle=False) as index:
        index_table = pd.DataFrame({
            'case_id': index['case_id'],
            'status': index['status'],
            'n_frames': index['n_frames'],
            'natoms': index['natoms'],
            **{label: index['fc_mean'][:, i] for i, label in enumerate(index['fc_labels'])},
            'output_path': index['output_path'],
            'error': index['error'],
        })
    display(index_table)
else:
    print('The full-dataset index does not exist yet.')

In [ ]:
# Example: load one completed case and make an MD-step table.
if master_index.exists():
    completed = index_table[index_table.status.isin(['completed', 'cached'])]
    if len(completed):
        selected_output = Path(completed.iloc[0].output_path)
        with np.load(selected_output, allow_pickle=False) as result:
            step_table = pd.DataFrame(
                result['fc_per_md_step'],
                index=result['step_ids'],
                columns=result['fc_labels'],
            ).rename_axis('md_step')
            display(step_table)
            print('Units:', result['fc_units'].item())
            print('Shell distances (angstrom):', result['shell_distances_ang'])